In [9]:
# 🚀 Step 1: Setup Environment and Pull Qwen2 model


import pandas as pd
from datasets import load_dataset
import networkx as nx
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
import ollama

# ✅ Pull the specific Qwen2 model
print("Pulling Qwen2 model...")

print("✅ Qwen2 model pulled successfully!")



Pulling Qwen2 model...
✅ Qwen2 model pulled successfully!


In [10]:
# Standard libraries
from datasets import load_dataset
import pandas as pd
import numpy as np
import re
import json
from collections import Counter

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Network
import networkx as nx

# LLM
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Optional: Community detection
try:
    import community as community_louvain
    HAS_LOUVAIN = True
except ImportError:
    HAS_LOUVAIN = False
    print("python-louvain not installed → skipping community detection")

# NLTK stopwords (English + Danish as per your spec)
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

print("Imports complete.")

python-louvain not installed → skipping community detection
Imports complete.


In [11]:
# Load the S&P 500 earnings transcripts dataset from Hugging Face
ds = load_dataset("kurry/sp500_earnings_transcripts", split="train")

print(f"Total transcripts in dataset: {len(ds):,}")
print(f"Columns: {list(ds[0].keys())}")

# Inspect first transcript
sample = ds[0]
print(f"\nExample: {sample['symbol']} | {sample['company_name']} | Q{sample['quarter']} {sample['year']}")
print(f"Number of speaker segments: {len(sample['structured_content'])}")
print("\nFirst segment:")
print(f"  Speaker: {sample['structured_content'][0]['speaker']}")
print(f"  Text (preview): {sample['structured_content'][0]['text'][:200]}...")

Total transcripts in dataset: 33,362
Columns: ['symbol', 'quarter', 'year', 'date', 'content', 'structured_content', 'company_name', 'company_id']

Example: A | Agilent Technologies, Inc. | Q4 2020
Number of speaker segments: 113

First segment:
  Speaker: Operator
  Text (preview): Good afternoon, and welcome to the Agilent Technologies Fourth Quarter Earnings Conference Call. All lines have been placed on mute to prevent any background noise. After the speakers' remarks, there ...


In [12]:
# Build a long-form DataFrame: one row per speaker utterance
rows = []

for t_id, rec in enumerate(ds):
    meta = {
        "transcript_id": t_id,
        "symbol": rec["symbol"],
        "company_name": rec["company_name"],
        "date": rec["date"],
        "quarter": rec["quarter"],
        "year": rec["year"],
    }
    for seg_id, seg in enumerate(rec["structured_content"]):
        rows.append({
            **meta,
            "seg_id": seg_id,
            "speaker": seg["speaker"],
            "text": seg["text"],
            "len_chars": len(seg["text"]),
        })

utterances = pd.DataFrame(rows)
utterances["date"] = pd.to_datetime(utterances["date"], errors="coerce")

print(f"Total utterances: {len(utterances):,}")
print(f"Unique companies: {utterances['company_name'].nunique():,}")
print(f"Year range: {utterances['year'].min()} → {utterances['year'].max()}")

utterances.head()

Total utterances: 2,794,981
Unique companies: 496
Year range: 2005 → 2025


,transcript_id,symbol,company_name,date,quarter,year,seg_id,speaker,text,len_chars
0,0,A,"Agilent Technologies, Inc.",2020-11-23 16:30:00,4,2020,0,Operator,"Good afternoon, and welcome to the Agilent Tec...",419
1,0,A,"Agilent Technologies, Inc.",2020-11-23 16:30:00,4,2020,1,Ankur Dhingra,"Thank you, and welcome everyone to Agilent's f...",1877
2,0,A,"Agilent Technologies, Inc.",2020-11-23 16:30:00,4,2020,2,Mike McMullen,"Thanks, Ankur, and thanks to everyone for join...",6890
3,0,A,"Agilent Technologies, Inc.",2020-11-23 16:30:00,4,2020,3,Bob McMahon,"Thanks, Mike, and good afternoon, everyone. In...",7245
4,0,A,"Agilent Technologies, Inc.",2020-11-23 16:30:00,4,2020,4,Ankur Dhingra,"Thanks, Bob. David, let's provide the instruct...",59


In [13]:
# Filter to 2024
d = utterances[utterances["year"] == 2024].copy().reset_index(drop=True)

print(f"Utterances in 2024: {len(d):,}")
print(f"Unique companies in 2024: {d['company_name'].nunique():,}")
print(f"Date range: {d['date'].min().date()} → {d['date'].max().date()}")

# Quick check: top speakers
print("\nTop 10 speaker types in 2024:")
print(d["speaker"].value_counts().head(10))

Utterances in 2024: 138,105
Unique companies in 2024: 493
Date range: 2023-05-17 → 2025-03-27

Top 10 speaker types in 2024:
speaker
Operator                25704
Unidentified Analyst      492
Jeremy Tonet              276
Nigel Coe                 238
Julian Mitchell           220
Deane Dray                211
Scott Davis               205
Rob Berkley               205
Gerard Cassidy            204
Ebrahim Poonawala         199
Name: count, dtype: int64


In [14]:
# Aggregate all utterances per company into one document
company_docs = (
    d.groupby(["symbol", "company_name"], as_index=False)
     .agg(
         full_text=("text", lambda x: " ".join(x.astype(str))),
         first_date=("date", "min"),
         last_date=("date", "max"),
         n_utterances=("text", "count")
     )
)

# Add word count
company_docs["word_count"] = company_docs["full_text"].str.split().str.len()

print(f"Company documents (2024): {len(company_docs)}")
print(f"Avg. words per company: {company_docs['word_count'].mean():.0f}")
print(f"Most verbose: {company_docs.loc[company_docs['word_count'].idxmax(), 'company_name']} "
      f"({company_docs['word_count'].max():,} words)")

company_docs = company_docs.sort_values("word_count", ascending=False).reset_index(drop=True)
company_docs.head()

Company documents (2024): 493
Avg. words per company: 31154
Most verbose: Equifax Inc. (65,011 words)


,symbol,company_name,full_text,first_date,last_date,n_utterances,word_count
0,EFX,Equifax Inc.,Greetings and welcome to the Equifax Corporate...,2024-04-18 08:30:00,2025-02-06 08:30:00,469,65011
1,C,Citigroup Inc.,"Hello, and welcome to Citi's Fourth Quarter 20...",2024-04-12 11:00:00,2025-01-15 11:00:00,333,50386
2,ENPH,"Enphase Energy, Inc.","Good afternoon, everyone and welcome to the En...",2024-04-23 16:30:00,2025-02-04 16:30:00,335,48923
3,DRI,"Darden Restaurants, Inc.",Welcome to the Darden Fiscal Year 2024 Fourth ...,2023-09-21 08:30:00,2024-06-20 08:30:00,457,46251
4,VZ,Verizon Communications Inc.,"Good morning, and welcome to the Verizon Fourt...",2024-04-22 10:00:00,2025-01-24 08:30:00,315,45994


In [15]:
# ------------------------------------------------------------
# Cell 7 – LLM Setup + Light Pre-processing (CPU only)
# ------------------------------------------------------------

# 1. Stop-words (English + Danish)
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

stop_en = set(stopwords.words("english"))
stop_da = set(stopwords.words("danish"))
STOP_WORDS = stop_en.union(stop_da)
print(f"Total stop-words: {len(STOP_WORDS)}")

# 2. Simple text cleaner (remove stop-words, keep >2-char words)
def clean_text(txt: str, max_chars: int = 1500) -> str:
    words = txt.lower().split()
    words = [w for w in words if w not in STOP_WORDS and len(w) > 2]
    out = " ".join(words)
    return out[:max_chars]                     # truncate for speed

# 3. Load a **tiny** Qwen model that runs on CPU
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

import ollama  # pip install ollama, and make sure the Ollama daemon is running

MODEL_NAME = "goekdenizguelmez/JOSIEFIED-Qwen3:latest"   # Ollama model tag

# 1. One-shot inference helper using Ollama
def qwen_infer(prompt: str, max_new: int = 120) -> str:
    """
    Call the local Ollama model and return the generated text.
    """
    resp = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={
            "temperature": 0.7,
            "num_predict": max_new,   # similar to max_new_tokens
        },
    )
    return resp["message"]["content"]

# Quick test
test_prompt = "List 3-5 short topics from this text in JSON: \"We grew revenue 12% thanks to AI products.\""
print("\nTest inference:")
print(qwen_infer(test_prompt)[:200])

Total stop-words: 285

Test inference:



In [16]:
# ------------------------------------------------------------
# Cell 7.5 – Reload & Filter to 2024 (Fix 'd' not defined)
# ------------------------------------------------------------

from datasets import load_dataset
import pandas as pd

print("Loading dataset...")
ds = load_dataset("kurry/sp500_earnings_transcripts", split="train")

# Flatten into utterances
rows = []
for t_id, rec in enumerate(ds):
    meta = {
        "transcript_id": t_id,
        "symbol": rec["symbol"],
        "company_name": rec["company_name"],
        "date": rec["date"],
        "quarter": rec["quarter"],
        "year": rec["year"],
    }
    for seg_id, seg in enumerate(rec["structured_content"]):
        rows.append({
            **meta,
            "seg_id": seg_id,
            "speaker": seg["speaker"],
            "text": seg["text"],
        })

utterances = pd.DataFrame(rows)
utterances["date"] = pd.to_datetime(utterances["date"], errors="coerce")

# Filter to 2024
d = utterances[utterances["year"] == 2024].copy().reset_index(drop=True)

print(f"Success! d created → {len(d):,} utterances from {d['company_name'].nunique()} companies in 2024")

Loading dataset...
Success! d created → 138,105 utterances from 493 companies in 2024


In [17]:
# ------------------------------------------------------------
# Cell 8 – Topic Discovery using Qwen2-1.5B (CPU)
# ------------------------------------------------------------

import random
import re
from collections import Counter

# 1. Sample 1,000 utterances from 2024
sample_utter = d.sample(n=1000, random_state=42).reset_index(drop=True)
print(f"Sampled {len(sample_utter):,} utterances for topic discovery")

# 2. Prompt template for topic extraction
TOPIC_PROMPT = """Extract 3–5 short, meaningful topics from this earnings-call snippet.
Return ONLY a JSON list like: ["topic 1", "topic 2", ...]
Do NOT add explanations.

Text:
{text}
"""

# 3. Run inference on each sample (progress bar)
print("\nRunning LLM on 1,000 samples (CPU – will take ~3–5 mins)...")
extracted_topics = []

for i, row in sample_utter.iterrows():
    if i % 200 == 0:
        print(f"  → Processed {i}/{len(sample_utter)}")

    clean_txt = clean_text(row["text"])
    if len(clean_txt) < 20:  # skip very short
        continue

    prompt = TOPIC_PROMPT.format(text=clean_txt)
    response = qwen_infer(prompt, max_new=80)

    # Extract JSON list with regex fallback
    match = re.search(r'\[\s*"(.*?)"\s*(?:,\s*"(.*?)")*\s*\]', response)
    if match:
        topics = [t.strip().lower() for t in match.groups() if t]
        extracted_topics.extend(topics)
    else:
        # Fallback: split by comma or newline
        fallback = [t.strip().lower() for t in re.split(r',|\n', response) if len(t) > 3]
        extracted_topics.extend(fallback[:5])

print(f"\nDone! Extracted {len(extracted_topics):,} raw topic phrases")

# 4. Deduplicate & rank by frequency
topic_counter = Counter(extracted_topics)
top_n = 60
unique_topics = [topic for topic, _ in topic_counter.most_common(top_n)]

print(f"\nTop {top_n} discovered topics (frequency):")
for t, cnt in topic_counter.most_common(10):
    print(f"  • {t:<25} ({cnt})")

print(f"\nFinal topic vocabulary size: {len(unique_topics)}")

Sampled 1,000 utterances for topic discovery

Running LLM on 1,000 samples (CPU – will take ~3–5 mins)...
  → Processed 0/1000
  → Processed 200/1000
  → Processed 400/1000
  → Processed 600/1000
  → Processed 800/1000

Done! Extracted 0 raw topic phrases

Top 60 discovered topics (frequency):

Final topic vocabulary size: 0
